In [1]:
import tensorflow as tf 
from tensorflow import keras 
import tensorflow_addons as tfa
import tensorflow_hub as hub 
from sklearn.metrics import mean_absolute_error 
import numpy as np 
import datetime
from deepface import DeepFace
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    Activation,
    Dense,
    Flatten,
    Conv2D,
    MaxPooling2D,
    AveragePooling2D,
    BatchNormalization,
    PReLU,  # <-- DITAMBAHKAN: Impor PReLU yang hilang
)
import os
import warnings

# Ignore irrelevant warning messages for cleaner output
warnings.filterwarnings('ignore')

c:\Users\ALFIAN\anaconda3\envs\env_insightface\lib\site-packages\tensorflow_addons\utils\tfa_eol_msg.py:23: UserWarning: 

TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

  warnings.warn(
c:\Users\ALFIAN\anaconda3\envs\env_insightface\lib\site-packages\tensorflow_addons\utils\ensure_tf_install.py:53: UserWarning: Tensorflow Addons supports using Python ops for all Tensorflow versions above or equal to 2.12.0 and strictly below 2.15.0 (nightly versions are not supported). 
 The versions of TensorFlow you are currently using is 2.15.0 and is not supported. 
Some things might work, some things might not.
If you were to encounter a bug, do no

Load Arcface


In [2]:
from deepface import DeepFace
import keras

# Load ArcFace model dari DeepFace
print("Loading ArcFace model...")
try:
    arcface_client = DeepFace.build_model('ArcFace')
    # Akses model Keras yang sebenarnya
    arcface_model = arcface_client.model
    print("✓ ArcFace model loaded successfully")
except Exception as e:
    print(f"✗ Error loading ArcFace model: {e}")
    raise

# Set model tidak dapat dilatih (frozen)
arcface_model.trainable = False

# Cetak summary ArcFace model
print("\n" + "="*80)
print("ARCFACE BASE MODEL ARCHITECTURE")
print("="*80)
arcface_model.summary()
print(f"\nArcFace Input Shape: {arcface_model.input_shape}")
print(f"ArcFace Output Shape: {arcface_model.output_shape}")
print(f"ArcFace Embedding Dimension: {arcface_model.output_shape[-1]}")

# Buat model transfer learning dengan ArcFace
print("\n" + "="*80)
print("BUILDING TRANSFER LEARNING MODEL WITH ARCFACE")
print("="*80)

try:
    inputs = keras.layers.Input(shape=(10, 112, 112, 3), name='Input')
    print(f"✓ Input layer created: {inputs.shape}")
    
    x = keras.layers.TimeDistributed(keras.layers.Rescaling(scale=1./255.0), name='Rescaling')(inputs)
    print(f"✓ Rescaling layer added: {x.shape}")
    
    x = keras.layers.TimeDistributed(arcface_model, name='ArcFace_TimeDistributed')(x)
    print(f"✓ ArcFace TimeDistributed layer added: {x.shape}")
    
    x = keras.layers.LSTM(units=128, return_sequences=True, name='LSTM_1')(x)
    print(f"✓ LSTM_1 layer added: {x.shape}")
    
    x = keras.layers.LSTM(units=64, name='LSTM_2')(x)
    print(f"✓ LSTM_2 layer added: {x.shape}")
    
    x = keras.layers.Dropout(0.2, name='Dropout_1')(x)
    print(f"✓ Dropout_1 layer added: {x.shape}")
    
    x = keras.layers.Dense(units=1024, name='Dense_1024')(x)
    print(f"✓ Dense_1024 layer added: {x.shape}")
    
    x = keras.layers.Dense(units=512, activation='relu', name='Dense_512')(x)
    print(f"✓ Dense_512 layer added: {x.shape}")
    
    x = keras.layers.Dense(256, activation='relu', name='Dense_256')(x)
    print(f"✓ Dense_256 layer added: {x.shape}")
    
    x = keras.layers.Dropout(0.5, name='Dropout_2')(x)
    print(f"✓ Dropout_2 layer added: {x.shape}")
    
    x = keras.layers.Dense(5, activation='sigmoid', name='Output')(x)
    print(f"✓ Output layer added: {x.shape}")
    
    model = keras.models.Model(inputs=inputs, outputs=x, name='ArcFace_Transfer_Learning')
    print("✓ Model compiled successfully\n")
    
except Exception as e:
    print(f"✗ Error building model: {e}")
    import traceback
    traceback.print_exc()
    raise

# Cetak summary model lengkap
print("\n" + "="*80)
print("COMPLETE TRANSFER LEARNING MODEL ARCHITECTURE")
print("="*80)
model.summary()

# Visualisasi model
print("\n" + "="*80)
print("GENERATING MODEL VISUALIZATION")
print("="*80)
try:
    keras.utils.plot_model(
        model, 
        to_file='arcface_transfer_learning_model.png', 
        show_shapes=True, 
        show_layer_names=True, 
        rankdir='TB', 
        expand_nested=True, 
        dpi=96
    )
    print("✓ Model diagram saved as 'arcface_transfer_learning_model.png'")
except Exception as e:
    print(f"✗ Error generating visualization: {e}")
    print("  (This is optional - model is still usable)")

# Informasi tambahan
print("\n" + "="*80)
print("MODEL INFORMATION SUMMARY")
print("="*80)

total_params = model.count_params()
trainable_params = sum([keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = sum([keras.backend.count_params(w) for w in model.non_trainable_weights])

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Non-trainable parameters (frozen ArcFace): {non_trainable_params:,}")
print(f"Percentage trainable: {(trainable_params/total_params)*100:.2f}%")

print(f"\nInput shape: {model.input_shape}")
print(f"Output shape: {model.output_shape}")
print(f"\nModel expects:")
print(f"  - Batch of 10 frames per sequence")
print(f"  - Each frame: 112x112x3 (RGB)")
print(f"  - Output: 5 classes with sigmoid activation")

print("\n" + "="*80)
print("LAYER-BY-LAYER OUTPUT SHAPES")
print("="*80)
for i, layer in enumerate(model.layers, 1):
    trainable_status = "Yes" if layer.trainable else "No"
    print(f"{i:2d}. {layer.name:35s} -> {str(layer.output_shape):30s} | Trainable: {trainable_status}")

# Informasi khusus ArcFace
print("\n" + "="*80)
print("ARCFACE MODEL DETAILS")
print("="*80)
print(f"ArcFace layers: {len(arcface_model.layers)}")
print(f"ArcFace parameters: {arcface_model.count_params():,}")
print(f"All ArcFace layers frozen: {not arcface_model.trainable}")

print("\n✓ Model setup complete and ready for training!")

Loading ArcFace model...
✓ ArcFace model loaded successfully

ARCFACE BASE MODEL ARCHITECTURE
Model: "ResNet34"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 112, 112, 3)]        0         []                            
                                                                                                  
 conv1_pad (ZeroPadding2D)   (None, 114, 114, 3)          0         ['input_1[0][0]']             
                                                                                                  
 conv1_conv (Conv2D)         (None, 112, 112, 64)         1728      ['conv1_pad[0][0]']           
                                                                                                  
 conv1_bn (BatchNormalizati  (None, 112, 112, 64)         256       ['conv1_conv[0][0]']        

### Load data

In [3]:
train_ds = tf.data.Dataset.load('C://Users//ALFIAN//TA CODING//ZIP FILE//ocean-project-deepface-20250529T023947Z-1-001//ocean-project-deepface//data//videoface//train_ds') \
    .cache().shuffle(buffer_size=1000, seed=42).prefetch(buffer_size=tf.data.AUTOTUNE)

valid_ds = tf.data.Dataset.load('C://Users//ALFIAN//TA CODING//ZIP FILE//ocean-project-deepface-20250529T023947Z-1-001//ocean-project-deepface//data//videoface//val_ds') \
    .cache().shuffle(buffer_size=1000, seed=42).prefetch(buffer_size=tf.data.AUTOTUNE)

train_ds, valid_ds

(<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 10, 112, 112, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 5), dtype=tf.float32, name=None))>,
 <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 10, 112, 112, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 5), dtype=tf.float32, name=None))>)

### Compile model

In [5]:
t = datetime.datetime.now().strftime("%m%d_%H%M%S")

# optimizer = keras.optimizers.Adam(learning_rate=0.001)
# optimizer = keras.optimizers.SGD(learning_rate=0.01, momentum=0.9)
# optimizer = tfa.optimizers.RectifiedAdam(learning_rate=0.001)

early_stopping = keras.callbacks.EarlyStopping(patience=10, verbose=1)
check_point    = keras.callbacks.ModelCheckpoint(filepath='./weights/faces/'+str(t)+'/face.t5',
                            monitor='val_mae',
                            mode='min',
                            save_best_only=True,
                            save_weights_only=True,
                            verbose=1)

model.compile(loss='mse', metrics=['mae'])

### Train

In [6]:
history = model.fit(train_ds, validation_data=valid_ds, batch_size=8, epochs=100, callbacks=[early_stopping, check_point])

Epoch 1/100
8/8 [==============================] - ETA: 0s - loss: 0.1020 - mae: 0.2516
Epoch 1: val_mae improved from inf to 0.02752, saving model to ./weights/faces/1010_052859\face.t5
8/8 [==============================] - 29s 3s/step - loss: 0.1020 - mae: 0.2516 - val_loss: 0.0010 - val_mae: 0.0275
Epoch 2/100
8/8 [==============================] - ETA: 0s - loss: 6.6428e-04 - mae: 0.0152
Epoch 2: val_mae improved from 0.02752 to 0.01508, saving model to ./weights/faces/1010_052859\face.t5
8/8 [==============================] - 22s 3s/step - loss: 6.6428e-04 - mae: 0.0152 - val_loss: 3.1880e-04 - val_mae: 0.0151
Epoch 3/100
8/8 [==============================] - ETA: 0s - loss: 4.5674e-04 - mae: 0.0128
Epoch 3: val_mae improved from 0.01508 to 0.00804, saving model to ./weights/faces/1010_052859\face.t5
8/8 [==============================] - 22s 3s/step - loss: 4.5674e-04 - mae: 0.0128 - val_loss: 9.5957e-05 - val_mae: 0.0080
Epoch 4/100
8/8 [==============================] - ETA: 

### Load weights

In [7]:
model.load_weights('./weights/faces/1010_052859/face.t5')

## Evaluation

### Training data

In [8]:
train_ds = tf.data.Dataset.load('C://Users//ALFIAN//TA CODING//ZIP FILE//ocean-project-deepface-20250529T023947Z-1-001//ocean-project-deepface//data//videoface//train_ds') 
loss, mae = model.evaluate(train_ds)
(1-mae)*100

8/8 [==============================] - 15s 2s/step - loss: 2.2177e-08 - mae: 1.0856e-04


99.98914429015713

In [9]:
y_true = np.concatenate([y for x,y in train_ds])
y_pred = model.predict(train_ds)

mae = mean_absolute_error(y_true, y_pred, multioutput='raw_values')
(1-mae)*100, (1-np.mean(mae))*100

8/8 [==============================] - 17s 2s/step


(array([99.99294, 99.9866 , 99.99015, 99.98764, 99.98839], dtype=float32),
 99.98914428870194)

### Validation data

In [10]:
valid_ds = tf.data.Dataset.load('C://Users//ALFIAN//TA CODING//ZIP FILE//ocean-project-deepface-20250529T023947Z-1-001//ocean-project-deepface//data//videoface//val_ds') 
loss, mae = model.evaluate(valid_ds)
(1-mae)*100

3/3 [==============================] - 5s 2s/step - loss: 9.4895e-08 - mae: 2.2202e-04


99.97779811092187

In [11]:
y_true = np.concatenate([y for x,y in valid_ds])
y_pred = model.predict(valid_ds)

mae = mean_absolute_error(y_true, y_pred, multioutput='raw_values')
(1-mae)*100, (1-np.mean(mae))*100

3/3 [==============================] - 5s 1s/step


(array([99.98497 , 99.97329 , 99.97999 , 99.974625, 99.97611 ],
       dtype=float32),
 99.97779811092187)

### Test data

In [12]:
test_ds = tf.data.Dataset.load('C://Users//ALFIAN//TA CODING//ZIP FILE//ocean-project-deepface-20250529T023947Z-1-001//ocean-project-deepface//data//videoface//test_ds')
loss, mae = model.evaluate(test_ds)
(1-mae)*100

3/3 [==============================] - 5s 2s/step - loss: 8.6166e-08 - mae: 2.0437e-04


99.97956252627773

In [13]:
y_true = np.concatenate([y for x,y in test_ds])
y_pred = model.predict(test_ds)

mae = mean_absolute_error(y_true, y_pred, multioutput='raw_values')
(1-mae)*100, (1-np.mean(mae))*100

3/3 [==============================] - 5s 1s/step


(array([99.98639, 99.97524, 99.98127, 99.9768 , 99.9781 ], dtype=float32),
 99.97956252627773)